# Experiment 3 · beryl fold (main run)

Based on `docs/PLAN.md` v16 §8.10-8.15. **One fold per session.** `EPOCHS` in (1) already holds the final epochs `B:20,Bp:20,C:50` (set after the 2026-09-18 pilot, PLAN §8.17); no need to change it.

**Fonts**: Colab usually has no CJK font; the script tries to install `fonts-noto-cjk` automatically (about 30-60 s the first time). If the log shows "[font] No CJK font, figure labels fall back to English", the figure labels are in English; this is normal and no tofu boxes appear.

Order: (1) (2) (3) (4) -> (5) start (background; configs already in the manifest are skipped) -> (6) watch progress (about 75-85 min) -> disconnect the session when done.
Resume after disconnect: after reconnecting run (1) (2) (3) (4) (5) (resumes automatically).

## (1) Settings

In [ ]:
FOLD = 'beryl'
DRIVE_ROOT = '/content/drive/MyDrive/Flood2'
PROJ = '/content/Flood_2.0'; DATA_DIR = f'{PROJ}/data/modelB_v1'
EXP2_ROOT = f'{DRIVE_ROOT}/models/exp2'          # Experiment-2 per-fold model_best.pt and norm_stats.json (starting point)
OUT_ROOT = f'{DRIVE_ROOT}/models'                  # Experiment-3 output -> OUT_ROOT/exp3/fold_<FOLD>/
FOLD_DIR = f'{OUT_ROOT}/exp3/fold_{FOLD}'
SAMPLES = f'{PROJ}/code/exp3_samples.csv'
EPOCHS = 'B:20,Bp:20,C:50'      # final epochs (set after the 2026-09-18 pilot, PLAN §8.17), no need to change
assert FOLD in ['irma','matthew','ian','milton','dorian','beryl']
print('FOLD =', FOLD, '| output', FOLD_DIR, '| epochs', EPOCHS)

## (2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for f in ['modelB_v1.tar','modelB_v1_manifest.csv','code/47_exp3_train.py','code/48_exp3_eval_lib.py','code/exp3_samples.csv', f'models/exp2/fold_{FOLD}/model_best.pt', f'models/exp2/fold_{FOLD}/norm_stats.json']:
    p = f'{DRIVE_ROOT}/{f}'; print(('OK  ' if os.path.exists(p) else 'MISSING ') + p)

## (3) Prepare code and data (copy -> unpack -> md5 check; skipped automatically if already done in this session)

In [ ]:
import os, shutil, subprocess, time
os.makedirs(f'{PROJ}/code', exist_ok=True); os.makedirs(f'{PROJ}/data', exist_ok=True)
for f in os.listdir(f'{DRIVE_ROOT}/code'):
    shutil.copy2(f'{DRIVE_ROOT}/code/{f}', f'{PROJ}/code/{f}')
print('code copied:', sorted(os.listdir(f'{PROJ}/code')))

ok_flag = f'{DATA_DIR}/.verified'
if os.path.exists(ok_flag):
    print('data already unpacked and verified, skipping')
else:
    t0 = time.time()
    shutil.copy2(f'{DRIVE_ROOT}/modelB_v1.tar', '/content/modelB_v1.tar')
    print('tar copied in %.0f s' % (time.time() - t0))
    tar_md5 = subprocess.run(['md5sum', '/content/modelB_v1.tar'], capture_output=True, text=True).stdout.split()[0]
    expect = open(f'{DRIVE_ROOT}/modelB_v1.tar.md5').read().split()[0]
    assert tar_md5 == expect, ('tar md5 mismatch', tar_md5, expect)
    print('tar md5 matches', tar_md5)
    subprocess.run(['tar', '-xf', '/content/modelB_v1.tar', '-C', f'{PROJ}/data'], check=True)
    os.remove('/content/modelB_v1.tar')
    r = subprocess.run(['python3', f'{PROJ}/code/41_pack_modelB_data.py', 'verify', '--data-dir', DATA_DIR,
                        '--manifest', f'{DRIVE_ROOT}/modelB_v1_manifest.csv'], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, 'per-file md5 check failed, stopping'
    open(ok_flag, 'w').write('ok\n')
    print('unpack + verification done, %.0f s in total' % (time.time() - t0))

## (4) Check GPU (confirm A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| python', platform.python_version())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU: change the runtime type to GPU and rerun from (1)')

## (5) Start / resume (background; guarded against double start)

In [ ]:
import os, subprocess, shlex, time
os.makedirs(FOLD_DIR, exist_ok=True); LOG = f'{FOLD_DIR}/train_stdout.log'
running = subprocess.run(['pgrep', '-f', f'[4]7_exp3_train.py --fold {FOLD} '], capture_output=True, text=True).stdout.split()
reason = None
if running:
    reason = f'This fold is already running in this session (PID {running}). Use (6) to watch progress.'
elif os.path.exists(LOG):
    age = time.time() - os.path.getmtime(LOG); txt = open(LOG, errors='replace').read()
    if 'fold complete' in txt: reason = 'The log already reports the fold as complete; not training again. You can disconnect.'
    elif age < 300: reason = f'The log was updated {age:.0f} s ago; another session may be running the {FOLD} fold. After checking, wait more than 5 minutes and rerun (resumes automatically from the manifest).'
if reason: print('!! Refusing to start:', reason)
else:
    cmd = (f'cd {shlex.quote(PROJ)} && nohup python3 -u code/47_exp3_train.py --fold {FOLD} --data-dir {shlex.quote(DATA_DIR)} '
           f'--exp2-root {shlex.quote(EXP2_ROOT)} --out-root {shlex.quote(OUT_ROOT)} --samples {shlex.quote(SAMPLES)} --epochs {EPOCHS} --eval-zero '
           f'>> {shlex.quote(LOG)} 2>&1 &')
    os.system(cmd); print('Started (configs already done are skipped per the manifest):', cmd); print('Run (6) in a few tens of seconds to watch progress; keep the page open')

## (6) Watch progress (rerun this cell to refresh)

In [ ]:
import csv, os, time, re
LOG = f'{FOLD_DIR}/train_stdout.log'; MP = f'{FOLD_DIR}/manifest.csv'
!pgrep -fl "[4]7_exp3_train.py --fold {FOLD} " || echo "not running (finished / disconnected)"
done = sum(1 for r in csv.DictReader(open(MP)) if r['status'] == 'done') if os.path.exists(MP) else 0
print(f'configs done {done} / 46 (incl. N=0); log updated {(time.time()-os.path.getmtime(LOG))/60:.0f} min ago' if os.path.exists(LOG) else 'no log yet')
print(''.join(open(LOG, errors='replace').readlines()[-6:]) if os.path.exists(LOG) else '')